# 01 -- The Data Model: SCI / ERR / DQ

Every science image in the pipeline is a **multi-extension FITS** file:

| Plane | HDU | Meaning |
|-------|-----|---------|
| `SCI` | primary | science data (electrons; Jy after flux calibration) |
| `ERR` | ext `ERR` | 1-sigma per-pixel uncertainty |
| `DQ`  | ext `DQ` | integer quality bitmask |

Phase 1 (notebook 02) is what actually turns raw frames into a calibrated
SCI/ERR/DQ file -- we have not run it yet. So here we first inventory the raw
frames on disk, then build a small **synthetic** SCI/ERR/DQ file with the same
three-plane structure to explore `read_mef` / `write_mef` and the DQ bitmask
right now. You will see the same mechanics again on a *real* calibrated frame
in notebook 02.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG  --  EDIT THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================
import os

# 1) Shared, read-only Astrometry.net index directory (used by Phase 2):
os.environ["CASSA_ASTROMETRY_INDEX"] = os.path.abspath("../../astrometry_data")

# 2) The provided workshop dataset + a writable work directory:
RAW_DIR   = "../raw"     # provided raw frames, one level up from notebooks/
WORK_DIR  = "../work"    # writable output dir, one level up from notebooks/

RAW_DIR   = os.path.abspath(os.path.expandvars(RAW_DIR))
WORK_DIR  = os.path.abspath(os.path.expandvars(WORK_DIR))

# These are relative to the working directory, which Jupyter sets to this
# notebook's folder. Fail loudly here rather than confusingly further down.
assert os.path.isdir(RAW_DIR), (
    f"RAW_DIR not found: {RAW_DIR}\nRun this notebook from the notebooks/ "
    f"directory, or set RAW_DIR/WORK_DIR to absolute paths above."
)

# Each phase writes into its own directory under WORK_DIR.
PHASE1_DIR = os.path.join(WORK_DIR, "phase1")   # calibrated frames
PHASE2_DIR = os.path.join(WORK_DIR, "phase2")   # master stacks + WCS
PHASE3_DIR = os.path.join(WORK_DIR, "phase3")   # flux-calibrated + catalogs
PHASE4_DIR = os.path.join(WORK_DIR, "phase4")   # diagnostics report
os.makedirs(WORK_DIR, exist_ok=True)
print("RAW_DIR    =", RAW_DIR)
print("PHASE1_DIR =", PHASE1_DIR)
print("PHASE2_DIR =", PHASE2_DIR)

## What Phase 1 will consume
Before running calibration (next notebook), inventory the raw frames by type
(science / flat / dark / bias) and check their FITS structure.

In [ ]:
import glob, os
from collections import Counter
from astropy.io import fits

raw_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.f*t*')))
print(f"Found {len(raw_files)} raw files in {RAW_DIR!r}\n")

def classify(img_type):
    img_type = (img_type or '').lower()
    if 'bias' in img_type: return 'bias'
    if 'dark' in img_type: return 'dark'
    if 'flat' in img_type: return 'flat'
    return 'science'  # matches ITelescopeNetworkProfile.get_image_type

type_counts = Counter()
breakdown = Counter()
n_hdus = Counter()
ext_names = Counter()
sci_files = []

for f in raw_files:
    with fits.open(f) as hdul:
        hdr = hdul[0].header
        kind = classify(hdr.get('IMAGETYP'))
        filt = hdr.get('FILTER', '<none>')
        exptime = hdr.get('EXPTIME', '<none>')
        type_counts[kind] += 1
        breakdown[(kind, str(filt), str(exptime))] += 1
        n_hdus[len(hdul)] += 1
        for hdu in hdul:
            ext_names[hdu.name] += 1
        if kind == 'science':
            sci_files.append(f)

print('Frame counts by type:')
for k in ('science', 'flat', 'dark', 'bias'):
    print(f'  {k:>8}: {type_counts.get(k, 0)}')

print('\nBreakdown by (type, filter, exptime):')
for k, v in sorted(breakdown.items()):
    print(f'  {k}: {v}')

print('\nHDU count per file (extensions present):')
for k, v in sorted(n_hdus.items()):
    print(f'  {k} HDU(s): {v} files')
print('Extension names seen:', dict(ext_names))

bpm_candidates = [f for f in raw_files if any(tag in os.path.basename(f).lower()
                                                for tag in ('badpix', 'bpm', 'mask'))]
print('\nStandalone bad-pixel-mask file present:', bool(bpm_candidates), bpm_candidates)
print('(cassa-photometry builds the bad-pixel mask on the fly from the master')
print(' flats during Phase 1 -- it is not a file you provide.)')

if sci_files:
    print(f"\nScience frame example: {os.path.basename(sci_files[0])}")
    with fits.open(sci_files[0]) as hdul:
        hdul.info()

## Build a toy SCI/ERR/DQ file
Phase 1 has not run yet, so there is no real calibrated frame on disk. To
explore the three-plane mechanics now, we fabricate a small synthetic image
-- a flat background, a handful of Gaussian "stars", Poisson + read noise for
`ERR`, and a few deliberately broken pixels (saturated, dead, cosmic-ray hit)
recorded in `DQ` -- and round-trip it through the pipeline's own `write_mef` /
`read_mef`. Notebook 02 will show the same reads on a *real* calibrated frame.

In [ ]:
import os
import numpy as np
from cassa_photometry.fits_utils import read_mef, write_mef, build_dq

rng = np.random.default_rng(0)
shape = (300, 300)

# Flat background + a handful of Gaussian "stars", in electrons.
yy, xx = np.mgrid[0:shape[0], 0:shape[1]]
background = 500.0
sci_clean = np.full(shape, background)
for _ in range(15):
    x0, y0 = rng.uniform(20, shape[1] - 20), rng.uniform(20, shape[0] - 20)
    amp = rng.uniform(2000, 20000)
    sigma = rng.uniform(1.5, 3.0)
    sci_clean += amp * np.exp(-(((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * sigma ** 2)))

read_noise = 5.0      # e-
saturation = 65000.0  # e-, the detector ceiling
sci = rng.poisson(sci_clean).astype(np.float32)

# Where the defects are ...
sat_y, sat_x = rng.integers(0, shape[0], 20), rng.integers(0, shape[1], 20)
saturated = np.zeros(shape, dtype=bool)
saturated[sat_y, sat_x] = True

bad_y, bad_x = rng.integers(0, shape[0], 15), rng.integers(0, shape[1], 15)
bad_pixel = np.zeros(shape, dtype=bool)
bad_pixel[bad_y, bad_x] = True

cr_y, cr_x = rng.integers(0, shape[0], 10), rng.integers(0, shape[1], 10)
cosmic_ray = np.zeros(shape, dtype=bool)
cosmic_ray[cr_y, cr_x] = True
cosmic_ray[sat_y[:3], sat_x[:3]] = True  # a few pixels carry two flags at once (DQ = 1|4 = 5)

# ... and what they actually do to the pixel values, so the DQ plane describes
# real defects rather than random positions.
sci[saturated] = saturation                                    # pegged at the ceiling
sci[cosmic_ray] = rng.uniform(2e4, 6e4, int(cosmic_ray.sum()))  # sharp spikes
sci[bad_pixel] = 0.0                                            # dead pixels

err = np.sqrt(sci + read_noise ** 2).astype(np.float32)  # Poisson + read noise, in quadrature
dq = build_dq(shape, saturated=saturated, bad_pixel=bad_pixel, cosmic_ray=cosmic_ray)

TOY_MEF = os.path.join(WORK_DIR, 'toy_sci_err_dq.fits')
write_mef(TOY_MEF, sci, err=err, dq=dq, history='Synthetic file for notebook 01 (no Phase 1 run required)')
print('Wrote', TOY_MEF)

sci, err, dq, hdr = read_mef(TOY_MEF)
print('SCI shape :', sci.shape, sci.dtype)
print('has ERR   :', err is not None)
print('has DQ    :', dq is not None)
print('median SCI:', float(np.nanmedian(sci)))


## Decode the DQ bitmask
A pixel may carry several flags at once (bitwise OR). We count each flag.

In [ ]:
from cassa_photometry.fits_utils import DQ_FLAG_NAMES
if dq is not None:
    total = dq.size
    for flag, name in DQ_FLAG_NAMES.items():
        n = int(np.count_nonzero(dq & flag))
        print(f'{name:>11}: {n:>8} px ({100*n/total:6.3f}%)')
else:
    print('No DQ plane in this file.')

## Visualise SCI / ERR / DQ side by side

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval
z = ZScaleInterval()
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
lo, hi = z.get_limits(np.nan_to_num(sci))
ax[0].imshow(sci, origin='lower', vmin=lo, vmax=hi, cmap='gray'); ax[0].set_title('SCI')
if err is not None:
    ax[1].imshow(err, origin='lower', cmap='magma'); ax[1].set_title('ERR (1-sigma)')
if dq is not None:
    ax[2].imshow(dq, origin='lower', cmap='tab10'); ax[2].set_title('DQ (bitmask)')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### Exercise 1 -- is the ERR plane really Poisson?
The `ERR` plane should roughly track $\sqrt{\text{signal}}$. Scatter `ERR`
against `SCI` for a random sample of pixels and check the trend. Then work out
*where* the read-noise floor matters: at what signal level does the
$\sqrt{S + \mathrm{RN}^2}$ term stop looking like plain $\sqrt{S}$?

In [ ]:
idx = np.random.default_rng(0).integers(0, sci.size, 5000)
sample_sci, sample_err = sci.ravel()[idx], err.ravel()[idx]

plt.figure(figsize=(6.5, 4.5))
plt.scatter(sample_sci, sample_err, s=3, alpha=0.3, label='pixels')
x = np.linspace(1, sample_sci.max(), 300)
plt.plot(x, np.sqrt(x), 'r--', lw=1.5, label=r'$\sqrt{\rm SCI}$  (pure Poisson)')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('SCI (e-)'); plt.ylabel('ERR (e-)')
plt.title('ERR tracks the square root of the signal')
plt.legend(); plt.tight_layout(); plt.show()

# Where does the read noise actually matter? Compare the two noise terms.
read_noise = 5.0
print(f"{'signal (e-)':>12} {'sqrt(S)':>10} {'sqrt(S+RN^2)':>14}   read-noise contribution")
for signal in (0, 25, 100, 500, 20000):
    poisson_only = np.sqrt(signal)
    with_read = np.sqrt(signal + read_noise ** 2)
    if poisson_only == 0:
        note = "read noise is the ONLY noise"
    else:
        note = f"inflates the error by {100 * (with_read / poisson_only - 1):.1f}%"
    print(f"{signal:>12} {poisson_only:>10.2f} {with_read:>14.2f}   {note}")

print("\nRead noise dominates in the dark, Poisson noise dominates on bright sources --")
print("which is why the ERR plane is not simply proportional to sqrt(SCI).")

### Exercise 2 -- decode the DQ bitmask by hand
List every distinct `DQ` value in the plane and translate it back into flag
names. Then count how many pixels are saturated, how many are cosmic rays, and
how many are **both** -- and check whether those per-flag counts add up to the
number of flagged pixels. (Watch the operator precedence: in Python `&` binds
tighter than `>`.)

In [ ]:
from cassa_photometry.fits_utils import (
    DQ_SATURATED, DQ_BAD_PIXEL, DQ_COSMIC_RAY, DQ_NO_DATA, DQ_FLAG_NAMES,
)

# Every distinct DQ value in the plane, decoded back into its flags.
values, counts = np.unique(dq[dq != 0], return_counts=True)
print(f"{'DQ':>4}  {'binary':>8}  {'pixels':>6}   flags")
print('-' * 52)
for v, c in zip(values, counts):
    names = [name for flag, name in DQ_FLAG_NAMES.items() if v & flag]
    print(f"{v:>4}  {v:>8b}  {c:>6}   {' | '.join(names)}")

# Test a single flag with bitwise AND: (dq & FLAG) > 0.
print(f"\nTotal flagged pixels : {int(np.count_nonzero(dq))}")
for flag, name in DQ_FLAG_NAMES.items():
    print(f"  {name:>11} (dq & {flag}) : {int(np.count_nonzero((dq & flag) > 0))}")

# CAREFUL: to ask "saturated AND cosmic ray", compare each flag to zero FIRST.
# (dq & 1) & (dq & 4) is always 0 -- those bits never overlap with each other.
both = ((dq & DQ_SATURATED) > 0) & ((dq & DQ_COSMIC_RAY) > 0)
print(f"\nSaturated AND cosmic ray  : {int(np.count_nonzero(both))}   <- correct")
print(f"(dq & 1) & (dq & 4) > 0   : {int(np.count_nonzero((dq & DQ_SATURATED) & (dq & DQ_COSMIC_RAY) > 0))}   <- wrong, always 0")

# How many flags does each flagged pixel carry?
n_flags = sum(((dq & flag) > 0).astype(int) for flag in DQ_FLAG_NAMES)
print(f"\nPixels with exactly one flag : {int(np.count_nonzero(n_flags == 1))}")
print(f"Pixels with more than one    : {int(np.count_nonzero(n_flags > 1))}  "
      f"(DQ values: {np.unique(dq[n_flags > 1])})")
print("\nThe per-flag counts sum to more than the number of flagged pixels,")
print("because a pixel flagged twice is counted in both rows.")


### Exercise 3 -- use DQ as a mask, and check the primary-HDU trick
Compute the frame statistics twice: over all pixels, and over only the
`DQ == 0` pixels. Which statistics move, and why? Finally, confirm the design
choice from the lecture: because `SCI` lives in the **primary** HDU, a tool
that knows nothing about this pipeline (plain `fits.getdata`) still reads the
science data correctly.

In [ ]:
from astropy.io import fits

good = dq == 0

print("Statistics computed over ALL pixels vs only DQ==0 pixels:")
print(f"{'':>10} {'all':>12} {'DQ==0 only':>12}")
for label, fn in (('median', np.median), ('mean', np.mean), ('max', np.max), ('min', np.min)):
    print(f"{label:>10} {fn(sci):12.2f} {fn(sci[good]):12.2f}")

lost = 100 * (1 - good.sum() / good.size)
print(f"\nMasking costs {good.size - good.sum()} pixels ({lost:.3f}% of the frame)")
print("The mean and max move a lot: saturated pixels and cosmic rays are")
print("exactly the outliers that would otherwise poison your photometry.")

# SCI lives in the PRIMARY HDU, so tools that know nothing about this pipeline
# still read the science data correctly.
plain = fits.getdata(TOY_MEF)
print(f"\nfits.getdata() returns the SCI plane unchanged: {np.allclose(plain, sci)}")
with fits.open(TOY_MEF) as hdul:
    print("Extensions in the file:", [(h.name, h.data.shape) for h in hdul])
